# Comparación grupal todos los codigos

## LOS 3 BLOQUES

Cada bloque responde a 4 preguntas y usa diferente variables, pero todos usan el mismo dataset.

Vamos a ver cada bloque en profundidad y al final vamos a compararlos y hacer una conclusion.

## Bloque 1 (modelo): Cross Validation

Resumen y detalles sobre este bloque. Que vamos a ver, a que responde y a que conclusion llegamos.

Preparación de datos:

In [ ]:
# Librerías de siempre
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Herramientas de scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats                  # para el contraste de significación

sns.set_theme(style="whitegrid")

In [ ]:
# El notebook está en models/, así que subimos una carpeta para llegar a datasets/
BASE = Path.cwd().parent
df = pd.read_csv(BASE / "datasets" / "melbourne_limpio.csv")

# Columnas de texto: habrá que convertirlas a números para los modelos lineales
CATEGORICAS = ["Type", "Suburb", "Method", "SellerG"]
print(f"Dimensiones: {df.shape[0]} viviendas x {df.shape[1]} columnas")
print(df.head())

In [ ]:
# Distribución del precio: si está sesgada, el RMSE lo dominarán las casas caras
plt.figure(figsize=(10, 6))
sns.histplot(df["Price"], kde=True, bins=50)
plt.title("Distribución del precio de venta", fontsize=16)
plt.xlabel("Precio (AUD)")
plt.ylabel("Frecuencia")
plt.show()

print(f"Mediana: {df['Price'].median():,.0f} AUD   "
      f"Máximo: {df['Price'].max():,.0f} AUD   "
      f"Asimetría: {df['Price'].skew():.2f}")

In [ ]:
# 1. Categóricas -> columnas 0/1
#    drop_first=True quita una categoría de cada columna: queda como referencia (si Type_t = 0 y Type_u = 0, es una casa)
df_modelo = pd.get_dummies(df, columns=CATEGORICAS, drop_first=True, dtype=int)

# 2. Separar X e y
X = df_modelo.drop("Price", axis=1)
y = df_modelo["Price"]
columnas = list(X.columns)   # guardamos los nombres para leer los coeficientes más adelante

# 3. PRIMERO se divide en train y test (con datos sin escalar)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# El scaler se ajusta SOLO con train y se aplica a ambos
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Entrenamiento: {X_train_s.shape[0]} viviendas")
print(f"Prueba:        {X_test_s.shape[0]} viviendas")
print(f"Columnas:      {X_train_s.shape[1]}")

# 4. Relación filas/columnas: cuantas más filas por columna, menos aporta la regularización (ejercicio 3)
ratio = X_train_s.shape[0] / X_train_s.shape[1]
print(f"\nRelación filas/columnas en entrenamiento: {ratio:.0f} a 1")

In [ ]:
# 13 valores de alpha entre 0,000001 y 1.000.000 (escala logarítmica)
alphas = np.logspace(-6, 6, 13)

# Diccionario con los modelos: así podemos recorrerlos todos con un bucle
modelos = {
    "LinearRegression": LinearRegression(),
    "RidgeCV": RidgeCV(alphas=alphas, scoring="neg_mean_squared_error"),       # no acepta random_state ni necesita cv
    "LassoCV": LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000),   # max_iter alto para que converja
}

Pregunta 1: Predice el precio medio de una vivienda

In [ ]:
import time

filas_k = []
for k in (3, 5, 10, 20):
    t0 = time.time()   # cronómetro: más folds = más entrenamientos = más tiempo
    scores = cross_val_score(LinearRegression(), X_train_s, y_train,
                             cv=KFold(k, shuffle=True, random_state=42), scoring="neg_mean_squared_error")
    rmse_k = np.sqrt(-scores)   # el error viene en negativo: le cambiamos el signo antes de la raíz
    filas_k.append({"K": k, "RMSE medio": rmse_k.mean(), "Desv. típica": rmse_k.std(),
                    "Desv./media (%)": 100 * rmse_k.std() / rmse_k.mean(), "Tiempo (s)": time.time() - t0})
pd.DataFrame(filas_k).set_index("K").round(2)

Pregunta 2: ¿Qué características encarecen una vivienda?

In [ ]:
# 5 columnas de ruido (misma semilla que en clase, así el experimento es reproducible)
np.random.seed(42)
X_train_r = np.hstack((X_train_s, np.random.rand(X_train_s.shape[0], 5)))
X_test_r = np.hstack((X_test_s, np.random.rand(X_test_s.shape[0], 5)))
columnas_r = columnas + [f"ruido_{i + 1}" for i in range(5)]

# Lasso y Ridge entrenados con el ruido incluido
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000).fit(X_train_r, y_train)
ridge_cv = RidgeCV(alphas=alphas, scoring="neg_mean_squared_error").fit(X_train_r, y_train)
print(f"Alpha Lasso: {lasso_cv.alpha_:,.6f}   Alpha Ridge: {ridge_cv.alpha_:,.6f}")

In [ ]:
# 1. Coeficientes del Lasso en una Series, indexada por nombre de variable
coefs = pd.Series(lasso_cv.coef_, index=columnas_r)

# 2. Cuántos coeficientes son cero
n_ceros = (coefs == 0).sum()
print(f"Coeficientes a cero: {n_ceros} de {len(coefs)}")

# 3. Qué ha pasado con las 5 variables de ruido
es_ruido = coefs.index.str.startswith("ruido_")   # máscara booleana: True solo en las 5 de ruido
coefs_ruido = coefs[es_ruido]                     # nos quedamos con los True
print("\nCoeficientes de las variables de ruido:")
print(coefs_ruido)
print(f"¿Las 5 de ruido están a cero? {(coefs_ruido == 0).all()}")

# 4. Variables reales que Lasso ha descartado (sin contar dummies de Suburb y SellerG, que son cientos)
coefs_reales = coefs[~es_ruido]   # ~ invierte la máscara: todas menos las de ruido
descartadas = coefs_reales[coefs_reales == 0].index.tolist()
print(f"\nVariables reales descartadas: {len(descartadas)}")
for nombre in [d for d in descartadas if not d.startswith(("Suburb_", "SellerG_"))]:
    print(f"  - {nombre}")
print(f"  + {sum(d.startswith('Suburb_') for d in descartadas)} barrios y "
      f"{sum(d.startswith('SellerG_') for d in descartadas)} inmobiliarias")

# 5. Top 10 de coeficientes positivos más grandes
top10 = coefs.sort_values(ascending=False).head(10)   # ordenada de mayor a menor

plt.figure(figsize=(10, 6))
sns.barplot(x=top10.values, y=top10.index, color="steelblue")
plt.title("Los 10 factores que más suben el precio (según Lasso)", fontsize=16)
plt.xlabel("Coeficiente (aumento del precio en AUD por cada desviación típica)")
plt.ylabel("")
plt.tight_layout()
plt.show()

Pregunta 3: ¿Qué tipo de vivienda genera más beneficio?

In [ ]:
# Solo viviendas de test con superficie construida real (BuildingArea_imputado = 0)
reales = X_test.index[df.loc[X_test.index, "BuildingArea_imputado"] == 0]
datos_13 = df.loc[reales].copy()
datos_13["precio_m2"] = datos_13["Price"] / datos_13["BuildingArea"]

# Indicador bruto: mediana por tipo
res_13 = datos_13.groupby("Type").agg(n=("Price", "size"), precio_mediano=("Price", "median"),
                                      m2_mediano=("BuildingArea", "median"), bruto_precio_m2=("precio_m2", "median"))

def matriz(datos):
    """Mismas columnas y escala que en el entrenamiento."""
    d = pd.get_dummies(datos.drop(columns="Price"), columns=CATEGORICAS, dtype=int)
    # reindex: añade con 0 las columnas que falten y quita las que sobren
    return scaler.transform(d.reindex(columns=columnas, fill_value=0))

for nombre in list(modelos):
    valores = {}
    for tipo in res_13.index:
        cambiado = datos_13.drop(columns="precio_m2").assign(Type=tipo)  # misma vivienda, solo cambia el tipo
        pred = modelos[nombre].predict(matriz(cambiado))
        valores[tipo] = np.median(pred / datos_13["BuildingArea"])   # precio predicho por m²
    res_13[f"controlado {nombre}"] = pd.Series(valores)

print("Tipo con mayor precio/m²:", {c: res_13[c].idxmax() for c in res_13.columns if "precio_m2" in c or "controlado" in c})
res_13.round(0)

Pregunta 4: ¿Es rentable un terreno sin construir?

In [ ]:
# Supuestos (cámbialos aquí)
COSTE_M2 = 2_000                               # AUD por m² construido
OTROS_COSTES = 0.15                            # 15 % extra sobre el coste de obra
SENSIBILIDAD = [1_500, 2_000, 2_500, 3_000, 3_500]   # costes a probar en la sensibilidad

# Solo casas (Type = h) con superficie construida real
casas = df[(df["Type"] == "h") & (df["BuildingArea_imputado"] == 0)].copy()
cat_14 = ["Suburb", "Method", "SellerG"]   # Type ya no hace falta: todas son casas

# Mismo proceso que antes: dummies + escalado, ahora solo con estas casas
d14 = pd.get_dummies(casas.drop(columns=["Price", "Type"]), columns=cat_14, drop_first=True, dtype=int)
columnas_14 = list(d14.columns)
scaler_14 = StandardScaler().fit(d14)

def matriz_14(datos):
    d = pd.get_dummies(datos.drop(columns=["Price", "Type"]), columns=cat_14, dtype=int)
    return scaler_14.transform(d.reindex(columns=columnas_14, fill_value=0))

X14, y14 = matriz_14(casas), casas["Price"]

# Los mismos tres modelos lineales, entrenados solo con las casas
modelos_14 = {
    "LinearRegression": LinearRegression(),
    "RidgeCV": RidgeCV(alphas=alphas, scoring="neg_mean_squared_error"),
    "LassoCV": LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000),
}
for m in modelos_14.values():
    m.fit(X14, y14)

# Predice con cualquiera de los tres modelos
def predecir_14(nombre, datos):
    return modelos_14[nombre].predict(matriz_14(datos))

def valor_m2(nombre, datos):
    """AUD que el modelo da a cada m² construido adicional (pendiente local, ±20 % de superficie)."""
    lo, hi = datos.copy(), datos.copy()
    lo["BuildingArea"] *= 0.8   # misma casa con un 20 % menos de superficie
    hi["BuildingArea"] *= 1.2   # ... y con un 20 % más
    # diferencia de precio / diferencia de m² = AUD por m²
    return (predecir_14(nombre, hi).mean() - predecir_14(nombre, lo).mean()) / (0.4 * datos["BuildingArea"].mean())

print(f"Casas con superficie construida real: {len(casas)}")

In [ ]:
# Región de cada barrio (no está en el CSV limpio)
region = pd.read_csv(BASE / "datasets" / "Melbourne_housing_FULL.csv").groupby("Suburb")["Regionname"].first()
casas["Regionname"] = casas["Suburb"].map(region)

filas = []
for nombre in modelos_14:
    # Primero todas las casas juntas y después región a región
    for zona, g in [("TODAS", casas)] + list(casas.groupby("Regionname")):
        g = g.drop(columns="Regionname")   # no es una variable del modelo
        b = valor_m2(nombre, g)
        terreno = np.median(predecir_14(nombre, g) - b * g["BuildingArea"])   # precio casa - valor de lo construido
        mercado = g["Price"].median()                                          # precio de mercado de la zona
        coste = g["BuildingArea"].median() * COSTE_M2 * (1 + OTROS_COSTES)     # construir una casa típica
        margen = mercado - terreno - coste                                     # > 0 -> rentable
        filas.append({"Modelo": nombre, "Zona": zona, "n": len(g), "valor_m2_construido": b,
                      "coste_max_m2_rentable": b / (1 + OTROS_COSTES), "terreno_estimado": terreno,
                      "precio_mercado": mercado, "terreno_%_precio": 100 * terreno / mercado,
                      "coste_construccion": coste, "margen_AUD": margen,
                      "margen_%_inversion": 100 * margen / (terreno + coste)})
res_14 = pd.DataFrame(filas).set_index(["Modelo", "Zona"])
res_14.round(0)

## Bloque 2: Random Forest

Resumen y detalles sobre este bloque. Que vamos a ver, a que responde y a que conclusion llegamos.

Preparación de datos:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

In [ ]:
from pathlib import Path

BASE = Path.cwd().parent
CLEAN_FILE = BASE / "datasets" / "melbourne_limpio.csv"
df = pd.read_csv(CLEAN_FILE)

df_modelo = pd.get_dummies(df, columns=["Type", "Suburb", "Method", "SellerG"], drop_first=True)
print(str(df_modelo.shape[0]) + "  " + str(df_modelo.shape[1]))
#print(df_modelo.head(4))
X = df_modelo.drop("Price", axis=1)
y = df_modelo["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"{len(df)} viviendas | {df.shape[1]} columnas")

In [ ]:
profundidades = [5, 10, 15, 20, 25, 30]
rmse_train, rmse_test = [], []

for d in profundidades:
    rf = RandomForestRegressor(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1).fit(X_train, y_train)
    rmse_train.append(np.sqrt(mean_squared_error(y_train, rf.predict(X_train))))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, rf.predict(X_test))))

plt.plot(profundidades, rmse_train, "o-", label="Entrenamiento")
plt.plot(profundidades, rmse_test, "o-", label="Prueba")
plt.xlabel("max_depth"); plt.ylabel("RMSE ($)"); plt.legend()
plt.show()

mejor_profundidad = profundidades[np.argmin(rmse_test)]
print(f"Mejor profundidad: {mejor_profundidad}")

In [ ]:
modelo_rf = RandomForestRegressor(n_estimators=300, max_depth=mejor_profundidad, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)
pred_test = modelo_rf.predict(X_test)

print(f"RMSE prueba: {np.sqrt(mean_squared_error(y_test, pred_test)):,.0f} $")
print(f"MAE prueba:  {mean_absolute_error(y_test, pred_test):,.0f} $")
print(f"R² prueba:   {r2_score(y_test, pred_test):.3f}")

# Validación cruzada: 5 entrenamientos con una parte distinta de prueba cada vez
r2_cv = cross_val_score(RandomForestRegressor(n_estimators=100, max_depth=mejor_profundidad, random_state=42, n_jobs=-1),
                        X, y, cv=5, scoring="r2")
print(f"R² en los 5 folds: {np.round(r2_cv, 3)} -> media {r2_cv.mean():.3f}")

Pregunta 1: Predice el precio medio de una vivienda

In [ ]:
print(f"Precio medio real:     {y_test.mean():,.0f} $")
print(f"Precio medio predicho: {pred_test.mean():,.0f} $")

plt.scatter(y_test, pred_test, alpha=0.3, s=8)
plt.plot([0, 4e6], [0, 4e6], color="red")
plt.xlim(0, 4e6); plt.ylim(0, 4e6)
plt.xlabel("Precio real ($)"); plt.ylabel("Precio predicho ($)")
plt.show()

Pregunta 2: ¿Qué características encarecen una vivienda?

In [ ]:
importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns)
importancias.index = [c.split("_")[0] if c.split("_")[0] in ["Type", "Suburb", "Method", "SellerG"] else c
                      for c in importancias.index]
importancias = importancias.groupby(level=0).sum().sort_values()

importancias.plot(kind="barh", figsize=(8, 5), title="Importancia de cada variable")
plt.show()

# ¿Sube o baja el precio? Correlación con el precio
df[["BuildingArea", "Distance", "YearBuilt", "Landsize", "Rooms", "Price"]].corr(method="spearman")["Price"].round(2)

Pregunta 3: ¿Qué tipo de vivienda genera más beneficio?

In [ ]:
prueba = df.loc[X_test.index].copy()
prueba["precio_m2"] = pred_test / prueba["BuildingArea"]
prueba = prueba[(prueba["BuildingArea_imputado"] == 0) & (prueba["BuildingArea"] >= 20)]

prueba.groupby("Type")[["precio_m2", "Price", "BuildingArea", "Landsize", "Distance"]].median().round(0)

Pregunta 4: ¿Es rentable un terreno sin construir?

In [ ]:
parcelas = df[(df["Type"] == "h") & (df["BuildingArea_imputado"] == 1) & (df["Landsize_imputado"] == 0) & (df["Landsize"] >= 200)].copy()

m2_nuevos = np.minimum(parcelas["Landsize"] * 0.5, 200)
coste = (parcelas["Price"] + m2_nuevos * 2200) * 1.12

# La misma parcela con una casa nueva
nueva = parcelas.assign(BuildingArea=m2_nuevos, YearBuilt=2026, BuildingArea_imputado=0).drop(columns="Price")
X_nueva = pd.get_dummies(nueva, columns=["Type", "Suburb", "Method", "SellerG"]).reindex(columns=X.columns, fill_value=0)

parcelas["margen_pct"] = (modelo_rf.predict(X_nueva) - coste) / coste * 100

print(f"{len(parcelas)} parcelas | margen mediano {parcelas['margen_pct'].median():.1f} % | rentables: {(parcelas['margen_pct'] > 15).mean() * 100:.1f} %")

por_suburbio = parcelas.groupby("Suburb")["margen_pct"].agg(["count", "median"])
por_suburbio[por_suburbio["count"] >= 5].sort_values("median", ascending=False).round(1).iloc[[0, 1, 2, 3, 4, -5, -4, -3, -2, -1]]

## Bloque 3: Catboost

Resumen y detalles sobre este bloque. Que vamos a ver, a que responde y a que conclusion llegamos.

Preparación de datos:

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#---------------------------------------preparamos-datos---------------------------------------

# 1. Definir la ruta dinámica al dataset limpio desde PRJ/models/
# Path().resolve() apunta a la carpeta 'models'. Su '.parent' es la carpeta 'PRJ'.
ROOT = Path().resolve().parent if Path().resolve().name == "models" else Path().resolve()
DATA_PATH = ROOT / "datasets" / "melbourne_limpio.csv"

# 2. Cargar el dataset
df = pd.read_csv(DATA_PATH)

# 3. Definir la variable objetivo y las variables categóricas
TARGET = "Price" #creamos variable de texto con el nombre de la columna que prediciremos
CAT_FEATURES = ["Type", "Suburb", "Method", "SellerG"] #creamos una lista con los nombres exactos de las 4 columnas que contienen palabras

# 4. Separar la matriz de características (X) y el objetivo (y)
X = df.drop(columns=[TARGET]) #creamos la columna eliminando la variable target, contiene las 14 columnas 
y = df[TARGET] #guarda los precios reales que el modelo debe aprender 

# Confirmación por pantalla
print("=" * 65)
print("  ESTADO DE LA CARGA DE DATOS")
print("=" * 65)
print(f"  Archivo cargado       : {DATA_PATH.name}")
print(f"  Dimensiones de X      : {X.shape[0]:,} filas x {X.shape[1]} columnas")
print(f"  Variables categóricas : {', '.join(CAT_FEATURES)}")
print("=" * 65 + "\n")

In [ ]:
from itertools import product
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---------------------------------------busqueda-hiperparametros---------------------------------------

# Probamos 6 combinaciones de profundidad (depth) y ritmo de aprendizaje (learning_rate)
rejilla = {
    "depth": [4, 6, 8],
    "learning_rate": [0.06, 0.12],
}

kf_grid = KFold(n_splits=3, shuffle=True, random_state=42)
resultados_grid = []

print("Buscando los mejores hiperparámetros para CatBoost...\n")
t0 = time.time()

for depth, lr in product(rejilla["depth"], rejilla["learning_rate"]):
    rmses_fold = []
    for train_idx, val_idx in kf_grid.split(X, y):
        m_tmp = CatBoostRegressor(
            iterations=500,
            depth=depth,
            learning_rate=lr,
            loss_function="RMSE",
            random_seed=42,
            verbose=0,
        )
        m_tmp.fit(X.iloc[train_idx], y.iloc[train_idx], cat_features=CAT_FEATURES)
        preds_tmp = m_tmp.predict(X.iloc[val_idx])
        rmses_fold.append(np.sqrt(mean_squared_error(y.iloc[val_idx], preds_tmp)))

    resultados_grid.append({
        "depth": depth,
        "learning_rate": lr,
        "RMSE_medio": np.mean(rmses_fold),
        "RMSE_desv": np.std(rmses_fold),
    })

df_grid = pd.DataFrame(resultados_grid).sort_values("RMSE_medio").reset_index(drop=True)
BEST_DEPTH = int(df_grid.loc[0, "depth"])
BEST_LR = float(df_grid.loc[0, "learning_rate"])

print(f"Búsqueda completada en {time.time() - t0:.1f} s.")
print(f"Mejor combinación -> depth = {BEST_DEPTH} | learning_rate = {BEST_LR} (RMSE: ${df_grid.loc[0, 'RMSE_medio']:,.0f})\n")
display(df_grid.round(2))

# Grafico justificativo de la seleccion de hiperparametros
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for lr, grupo in df_grid.sort_values("depth").groupby("learning_rate"):
    estilo = "o-" if lr == BEST_LR else "s--"
    color = "#d97724" if lr == BEST_LR else "#7f8c8d"
    ax.plot(grupo["depth"], grupo["RMSE_medio"], estilo, color=color, linewidth=2, markersize=7, label=f"learning_rate = {lr}")

ax.set_title("Selección de Hiperparámetros en CatBoost (Validación Cruzada)", fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("Profundidad del árbol (depth)", fontsize=10.5)
ax.set_ylabel("RMSE Medio (AUD) - Menor es mejor", fontsize=10.5)
ax.set_xticks(rejilla["depth"])
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

Pregunta 1: ¿Podemos predecir el precio de una vivienda en Melbourne con una precisión razonable mediante CatBoost y validación cruzada (
-fold)?

In [ ]:
#---------------------------------------dividimos-datos-entreno---------------------------------------

# 1. Configurar K-Fold con 5 particiones aleatorias pero reproducibles
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#hacemos 5 grupos diferentes

# 2. Listas para guardar las métricas calculadas en cada uno de los 5 folds
rmse_scores = []
mae_scores = []
r2_scores = []
feature_importances_list = [] #Guardará el peso de las variables de cada uno de los 5 folds
#listas vacias donde guardaremos las metricas

# Vector para guardar las predicciones Out-Of-Fold de todo el dataset
oof_preds = np.zeros(len(df)) #con esta linea creamos llena de zeros con 27.246 posiciones, una por cada casa 

print("Iniciando Validación Cruzada (5-Fold) con CatBoost...\n")

# 3. Bucle principal de validación cruzada
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1): # generamos las 5 parejas y cada una 80% y 20% 
    # Separar datos en entrenamiento (80%) y validación (20%) para este fold
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx] 
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx] 
    # dividimos bloque de entrenamiento y bloque de validacion

    # Instanciar el modelo CatBoost con los mejores hiperparámetros
    model = CatBoostRegressor(
        iterations=500, # 500 arboles
        learning_rate=BEST_LR, #el ritmo que tendra, cada arbol corrige un 12%
        depth=BEST_DEPTH, #profundidad de 6 por arbol
        loss_function="RMSE",
        random_seed=42,
        verbose=0  # Mantiene la pantalla limpia ocultando el progreso árbol por árbol
    )

    # Entrenar el modelo pasándole las variables categóricas
    model.fit(X_train, y_train, cat_features=CAT_FEATURES)
    #entrenamos el modelo, le especificmaos las que son de texto para que aplique su algortimos interno

    # Predecir sobre las viviendas del bloque de validación
    preds = model.predict(X_val)# le pasamos las 14 cracteristicas de las casas del 20% sin precios
    
    # Guardar las predicciones de este 20% en su posición real del dataset
    oof_preds[val_idx] = preds #en la ronda 1 del kfold esta examina 20% (5.449 casas) de las casa y saca las predicciones, coge esos 5.449 predicciones y los guarda exactamente en las filas correcpondientes 
    #val_index dentro de la libreta que creamos antes, en la ronda 5 la libreta oof_preds habra rellenado el 100% de las casas 

    # Calcular métricas del fold actual
    rmse = np.sqrt(mean_squared_error(y_val, preds)) #compara precios reales con los predichos rmse
    mae = mean_absolute_error(y_val, preds) #compara la diferencia absoluta directa entre lo predich y realidad
    r2 = r2_score(y_val, preds) #compara la precision del modelo frente a una prediccion basica de la media 

    # Guardar métricas en las listas acumuladoras
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    feature_importances_list.append(model.get_feature_importance())  #nos devuelve un listado con el porcentaje por columa, en total suman un 100% 

    print(f"  [Fold {fold}/5] RMSE: ${rmse:>10,.2f} | MAE: ${mae:>9,.2f} | R2: {r2:.4f}")

# 4. Mostrar resumen global de resultados
print("\n" + "=" * 65)
print("     RESULTADOS GLOBALES DE CATBOOST (MEDIA +/- DESVIACION ESTANDAR)")
print("=" * 65)
print(f"  RMSE Medio : ${np.mean(rmse_scores):>10,.2f}  ( +/- ${np.std(rmse_scores):,.2f} )")
print(f"  MAE Medio  : ${np.mean(mae_scores):>10,.2f}  ( +/- ${np.std(mae_scores):,.2f} )")
print(f"  R2 Medio   : {np.mean(r2_scores):>11.4f}  ( +/- {np.std(r2_scores):.4f} )")
print("=" * 65)

Pregunta 2: ¿Qué características encarecen o devalúan una vivienda según CatBoost?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#---------------------------------------2.2-importancia-variables---------------------------------------

# 1. Calcular el porcentaje medio de importancia promediando los 5 folds del 2.1
mean_importances = np.mean(feature_importances_list, axis=0) #hacemos la media de los 5 modelos entrenados

# 2. Crear una tabla pandas ordenada de mayor a menor importancia
df_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": mean_importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

# 3. Imprimir por pantalla la tabla en texto plano alineada
print("=" * 65)
print("     IMPORTANCIA MEDIA DE VARIABLES EN CATBOOST (5-FOLD CV)")
print("=" * 65)
for idx, row in df_importance.iterrows():
    print(f"  {idx+1:>2}. {row['Feature']:<25} : {row['Importance']:>6.2f} %")
print("=" * 65)

# 4. Generar el gráfico de barras horizontal
plt.figure(figsize=(10, 6))
plt.barh(df_importance["Feature"][::-1], df_importance["Importance"][::-1], color="#2b5c8f")
plt.xlabel("Importancia Relativa Media (%)")
plt.ylabel("Características (Features)")
plt.title("Factores que más influyen en el precio de la vivienda (CatBoost - 5 Folds)", fontsize=12, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

Pregunta 3: ¿Qué tipo de vivienda (h = casa, u = piso/unidad, t = adosado) resulta más beneficiosa?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#---------------------------------------2.3-beneficio-por-tipo---------------------------------------

#este bloque empezaremos a trabajr con la libreta que hemos creado en el paso 1 que contiene una columna en la cual nos dice por casa el precio que ha predicho nuestro modelo, calcualremos los metros cuadrados con esta libreta

# 1. Copiamos el dataframe y añadimos las predicciones OOF guardadas del 2.1
df_beneficio = df.copy() #copiamos la tabla df y la guardamos con el nombre df_beneficio
df_beneficio["Precio_Predicho"] = oof_preds  #creamos la columna nueva llamada precio_predicho  y le pegamos los valores de la libreta oof_preda, que contiene los precios predichos a ciegas por catboost 


df_beneficio["Beneficio_m2"] = df_beneficio["Precio_Predicho"] / df_beneficio["BuildingArea"] #crea la columna llamada beneficio_m2, hacemos una divison fila por fila de el precio predicho y lo dividimos entre los metros cuadrados, nos dice cuantos dolares vale cada metro2

# 3. Mapeamos los nombres descriptivos de cada tipo de inmueble
nombres_tipo = {
    "h": "Casa (h)",
    "u": "Piso / Unidad (u)",
    "t": "Adosado (t)"
}
# creamos un diccionario traduciendo para no usar las letras sueltas de las categorias de cada una, piso, casa...

df_beneficio["Tipo_Nombre"] = df_beneficio["Type"].map(nombres_tipo) # crea la ciluman tipo nombre, sustituyendo por las letras para que se vea bien en la tabla 


# 4. Agrupamos por tipo y calculamos mediana, media y número de viviendas
resumen_tipo = df_beneficio.groupby("Tipo_Nombre")["Beneficio_m2"].agg( #separamos las casas en 3 grupos, segun la categoria, sobre la col beneficios_m2 calculamos 3 datos
    Mediana_m2="median", # la mediana por el precio
    Media_m2="mean", #media aritmetica haitual 
    Total_Viviendas="count" #cuantas casas de cada tipo
).reset_index().sort_values(by="Mediana_m2", ascending=False) #lo convertimos en una tabla de pandas y ordenamos la tabla para que la metodologia mas importantye salga la primera 


#la mediana se calcula, ordenando todos los datos de mayor a menor y nos quedamos con el dato que deja el 50% de las casa por debajo y el 50% de las casas arriba para coger el de la mitas
#la media se calcula segun sumando todos los valores y los dividimos entre el numero que hay 




# 5. Imprimimos la tabla de resultados alineada por consola
print("=" * 65)
print("     INDICADOR DE BENEFICIO PROXY ($ / m2 CONSTRUIDO)")
print("=" * 65)
for idx, row in resumen_tipo.iterrows():
    print(f"  {row['Tipo_Nombre']:<20} | Mediana: ${row['Mediana_m2']:>8,.2f}/m² | Media: ${row['Media_m2']:>8,.2f}/m² | N = {row['Total_Viviendas']:,}")
print("=" * 65)

#Recorre la tabla resumen_tipo fila a fila e imprime un cuadro de texto limpio con los resultados formateados en dólares por $m^2$ y alineados verticalmente


# 6. Generamos el gráfico de barras comparativo
plt.figure(figsize=(8, 5))
bars = plt.bar(resumen_tipo["Tipo_Nombre"], resumen_tipo["Mediana_m2"], color=["#2b5c8f", "#d95f02", "#7570b3"])
plt.ylabel("Beneficio Mediano ($ / m² construido)")
plt.title("Beneficio Proxy por m² según Tipología de Inmueble (CatBoost)", fontsize=11, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.6)

# Añadimos el valor exacto sobre cada barra
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"${yval:,.2f}/m²", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

Pregunta 4: ¿Es rentable adquirir un terreno vacante sin construir para edificar?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from IPython.display import display

# ---------------------------------------2.4-paso-a-supuestos-y-modelo-catboost---------------------------------------

# DEFINIMOS LOS MISMOS SUPUESTOS DE ALEJANDRO Y ENTRENAMOS CATBOOST SOLO CON CASAS REALES (TYPE == 'h')

# 1. Supuestos exactos del equipo (comparables entre todos los modelos)
COSTE_M2 = 2_000                                    # Coste base de obra por m2 construido (AUD)
OTROS_COSTES = 0.15                                 # 15% extra sobre obra (permisos, tasas, imprevistos y venta)
SENSIBILIDAD = [1_500, 2_000, 2_500, 3_000, 3_500]  # Escenarios de coste/m2 para el grafico final

# 2. Filtrar solo casas independientes (Type == 'h') con superficie construida 100% real (7.889 filas)
casas = df[(df["Type"] == "h") & (df["BuildingArea_imputado"] == 0)].copy()

# 3. Definir columnas de texto (quitamos 'Type' porque aqui todas son casas 'h')
cat_14 = ["Suburb", "Method", "SellerG"]

# 4. Separar caracteristicas (X14) y precio objetivo (y14)
X14 = casas.drop(columns=["Price", "Type"])
y14 = casas["Price"]

# 5. Entrenar modelo CatBoost exclusivo para esta valoracion
model_terrenos = CatBoostRegressor(
    iterations=500,
    learning_rate=BEST_LR,
    depth=BEST_DEPTH,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)
model_terrenos.fit(X14, y14, cat_features=cat_14)

# 6. Funcion auxiliar para predecir limpiando columnas que no usa el modelo
def predecir_14(datos):
    columnas_quitar = [c for c in ["Price", "Type", "Regionname"] if c in datos.columns]
    return model_terrenos.predict(datos.drop(columns=columnas_quitar))

# 7. Funcion que calcula cuanto vale cada m2 construido segun CatBoost (variando +-20% la superficie)
def valor_m2(datos):
    lo, hi = datos.copy(), datos.copy()
    lo["BuildingArea"] *= 0.8   # Misma casa con un 20% menos de metros construidos
    hi["BuildingArea"] *= 1.2   # Misma casa con un 20% mas de metros construidos
    return (predecir_14(hi).mean() - predecir_14(lo).mean()) / (0.4 * datos["BuildingArea"].mean())

print("=" * 65)
print("     MODELO CATBOOST ENTRENADO PARA TERRENOS (PASO A)")
print("=" * 65)
print(f"  Casas con superficie construida real : {len(casas):,}")
print(f"  Valor medio estimado por m2 (CatBoost): ${valor_m2(casas):,.2f} AUD/m2")
print("=" * 65)


# ---------------------------------------2.4-paso-b-rentabilidad-por-regiones---------------------------------------

# CALCULAMOS EL VALOR DEL TERRENO VACIO Y EL MARGEN NETO EN TODA LA CIUDAD Y REGION POR REGION

# 1. Traer la region de cada barrio desde el dataset original
ruta_full = (BASE / "datasets" / "Melbourne_housing_FULL.csv") if "BASE" in globals() else Path("../datasets/Melbourne_housing_FULL.csv")
region = pd.read_csv(ruta_full).groupby("Suburb")["Regionname"].first()
casas["Regionname"] = casas["Suburb"].map(region)

# 2. Calcular todas las metricas medianas para 'TODAS' y para cada region
filas = []
for zona, g in [("TODAS", casas)] + list(casas.groupby("Regionname")):
    g = g.drop(columns="Regionname")
    b = valor_m2(g)
    terreno = np.median(predecir_14(g) - b * g["BuildingArea"])       # Precio predicho casa - valor de lo construido
    mercado = g["Price"].median()                                      # Precio mediano de mercado en la zona
    coste = g["BuildingArea"].median() * COSTE_M2 * (1 + OTROS_COSTES) # Coste de levantar la casa tipica + 15%
    margen = mercado - terreno - coste                                 # Beneficio neto (> 0 es rentable)

    filas.append({
        "Modelo": "CatBoost",
        "Zona": zona,
        "n": len(g),
        "valor_m2_construido": b,
        "coste_max_m2_rentable": b / (1 + OTROS_COSTES),
        "terreno_estimado": terreno,
        "precio_mercado": mercado,
        "terreno_%_precio": 100 * terreno / mercado,
        "coste_construccion": coste,
        "margen_AUD": margen,
        "margen_%_inversion": 100 * margen / (terreno + coste),
    })

res_14 = pd.DataFrame(filas).set_index(["Modelo", "Zona"])
display(res_14.round(0))


# ---------------------------------------2.4-paso-c-sensibilidad-coste-obra---------------------------------------

# SIMULAMOS COMO CAMBIA EL MARGEN (%) SI CONSTRUIR CUESTA ENTRE 1.500 Y 3.500 AUD/M2

todas = casas.drop(columns="Regionname")
b = valor_m2(todas)
terreno = np.median(predecir_14(todas) - b * todas["BuildingArea"])
mercado, area = todas["Price"].median(), todas["BuildingArea"].median()

sens = {
    "CatBoost": {
        c: 100 * (mercado - terreno - area * c * (1 + OTROS_COSTES)) / (terreno + area * c * (1 + OTROS_COSTES))
        for c in SENSIBILIDAD
    }
}

sens = pd.DataFrame(sens).rename_axis("AUD/m2")

# Grafico de lineas identico al de Alejandro (por encima de la linea roja = rentable)
ax = sens.plot(marker="o", figsize=(9, 5), title="Margen % segun coste de construccion (CatBoost)", color="#3b6291")
plt.axhline(0, color="red", linestyle="-", linewidth=1.2)
plt.ylabel("Margen (%)")
plt.xlabel("Coste de construccion (AUD/m2)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

display(sens.round(1))


# ---------------------------------------2.4-grafico-visual-regiones---------------------------------------

import matplotlib.ticker as ticker

orden_regiones = [
    "Western Victoria",
    "Northern Victoria",
    "Eastern Victoria",
    "Western Metropolitan",
    "Northern Metropolitan",
    "South-Eastern Metropolitan",
    "Eastern Metropolitan",
    "Southern Metropolitan",
]

df_reg = res_14.loc["CatBoost"]
global_aud = df_reg.loc["TODAS", "margen_AUD"]
global_pct = df_reg.loc["TODAS", "margen_%_inversion"]

df_plot = df_reg.reindex(orden_regiones).dropna()
colores = ["#2e8b57" if val > 0 else "#c85a17" for val in df_plot["margen_AUD"]]

fig, ax = plt.subplots(figsize=(11.5, 6))
barras = ax.barh(df_plot.index, df_plot["margen_AUD"], color=colores, height=0.62, edgecolor="black", linewidth=0.7)

ax.axvline(0, color="black", linewidth=1.1)
ax.set_xlim(-480_000, 340_000)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

ax.set_title(
    f"Actividad 4 - Margen Neto Mediano por Región al Edificar: CatBoost\n"
    f"(Global Melbourne: {'+' if global_aud >= 0 else '-'}${abs(global_aud):,.0f} AUD | {global_pct:+.1f}%)",
    fontsize=12.5, fontweight="bold", pad=12
)
ax.set_xlabel("Margen Neto Mediano (AUD) y Rentabilidad sobre Inversión (%)", fontsize=10.5)

for barra, (_, fila) in zip(barras, df_plot.iterrows()):
    val = fila["margen_AUD"]
    pct = fila["margen_%_inversion"]
    signo = "+" if val >= 0 else "-"
    texto = f"{signo}${abs(val):,.0f} ({pct:+.1f}%)"
    offset = 10_000 if val >= 0 else -10_000
    alineacion = "left" if val >= 0 else "right"
    ax.text(
        val + offset,
        barra.get_y() + barra.get_height() / 2,
        texto,
        va="center", ha=alineacion, fontsize=9.5, fontweight="bold"
    )

plt.tight_layout()
plt.show()

## Comparación

Una celda de 3 graficos de la misma pregunta.

In [ ]:
#celda para comparación